In [4]:
import pandas as pd
from sklearn.model_selection import train_test_split
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification
import torch

df = pd.read_csv("../data/dataset.csv")

df['label_id'] = df['label'].map({'safe': 0, 'injection': 1})

train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])

print(f"Train: {len(train_df)} | Test: {len(test_df)}")
print(f"\nTrain distribution:\n{train_df['label'].value_counts()}")
print(f"\nTest distribution:\n{test_df['label'].value_counts()}")

Train: 396 | Test: 99

Train distribution:
label
safe         199
injection    197
Name: count, dtype: int64

Test distribution:
label
safe         50
injection    49
Name: count, dtype: int64


In [ ]:
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
def tokenize(texts, max_length=128):
    return tokenizer(
        list(texts),
        padding=True,
        truncation=True,
        max_length=max_length,
        return_tensors='pt'
    )
train_encodings = tokenize(train_df['text'])
test_encodings = tokenize(test_df['text'])

print("Tokenization complete.")
print(f"Train input shape: {train_encodings['input_ids'].shape}")
print(f"Test input shape: {test_encodings['input_ids'].shape}")

Tokenization complete.
Train input shape: torch.Size([396, 27])
Test input shape: torch.Size([99, 26])


In [7]:
import torch
from torch.utils.data import Dataset, DataLoader

class PromptDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

train_labels = list(train_df['label_id'])
test_labels = list(test_df['label_id'])

train_dataset = PromptDataset(train_encodings, train_labels)
test_dataset = PromptDataset(test_encodings, test_labels)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16)

print(f"Train batches: {len(train_loader)}")
print(f"Test batches: {len(test_loader)}")

Train batches: 25
Test batches: 7


In [8]:
from transformers import DistilBertForSequenceClassification
from torch.optim import AdamW

model = DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=2)

optimizer = AdamW(model.parameters(), lr=2e-5)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
print(f"Training on: {device}")

# Training loop
EPOCHS = 3

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for batch in train_loader:
        optimizer.zero_grad()
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {avg_loss:.4f}")

print("\nTraining complete.")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Training on: cpu
Epoch 1/3 | Loss: 0.4792
Epoch 2/3 | Loss: 0.1002
Epoch 3/3 | Loss: 0.0536

Training complete.


In [9]:
from sklearn.metrics import classification_report, confusion_matrix

model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        preds = torch.argmax(outputs.logits, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

print(classification_report(all_labels, all_preds, target_names=['safe', 'injection']))
print("Confusion Matrix:")
print(confusion_matrix(all_labels, all_preds))

              precision    recall  f1-score   support

        safe       1.00      1.00      1.00        50
   injection       1.00      1.00      1.00        49

    accuracy                           1.00        99
   macro avg       1.00      1.00      1.00        99
weighted avg       1.00      1.00      1.00        99

Confusion Matrix:
[[50  0]
 [ 0 49]]


In [10]:
import os

os.makedirs("../models", exist_ok=True)
model.save_pretrained("../models/piguard_distilbert")
tokenizer.save_pretrained("../models/piguard_distilbert")

print("Model saved to models/piguard_distilbert")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to models/piguard_distilbert
